# Part F — evaluation on the official TEST set (B1, B2, D1, D2)

Nothing is trained or tuned on test. For each approach the **best configuration on val** is read from `results/results.csv`
and scored on the 3 000 test tweets with the **threshold chosen on val**.

- **B1 / B2**: pretrained models, nothing to load — they simply score the test tweets.
- **D1 / D2**: the fitted models saved by the validation notebooks (`models/`). If a file is missing, only that configuration is retrained
  on train (early-stopped on val) and saved; its val metrics are then recomputed so val and test come from the same model.

Output: `test/results_test.csv` (val + test metrics side by side, `source` = pretrained / loaded / retrained) and
`test/predictions/*.csv` (for McNemar / bootstrap). `results/results.csv` is **never modified**.

In [1]:
import os
import sys

REPO = "Arabic-sarcasm"

if not os.path.exists(f"/content/{REPO}"):
    !git clone https://github.com/cad0m/{REPO}.git /content/{REPO}
else:
    !git -C /content/{REPO} pull

%cd /content/{REPO}

if f"/content/{REPO}" not in sys.path:
    sys.path.insert(0, f"/content/{REPO}")

!pip install -q uv
!uv pip install --system emoji tashaphyne

Cloning into '/content/Arabic-sarcasm'...
remote: Enumerating objects: 136, done.
remote: Counting objects: 100% (136/136), done.
remote: Compressing objects: 100% (90/90), done.
remote: Total 136 (delta 73), reused 81 (delta 33), pack-reused 0 (from 0)
Receiving objects: 100% (136/136), 808.22 KiB | 3.18 MiB/s, done.
Resolving deltas: 100% (73/73), done.
/content/Arabic-sarcasm
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.3/17.3 MB 86.7 MB/s eta 0:00:00
Using Python 3.13.15 environment at: /usr
Resolved 4 packages in 231ms
Prepared 3 packages in 48ms
Installed 3 packages in 2ms
 + emoji==2.16.0
 + pyarabic==0.6.15
 + tashaphyne==0.3.6


In [2]:
# arabert pins an old emoji (1.x, no Arabic names): install it without deps, then force emoji>=2.16.
# If you see a KeyError in emoji.demojize, restart the runtime once and re-run from the top.
%pip install -q --no-deps arabert
%pip install -q -U "emoji>=2.16" pyarabic sentencepiece protobuf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.2/343.2 kB 8.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 7.36.2 which is incompatible.


In [3]:
import copy
import time

import joblib
import numpy as np
import pandas as pd
import torch
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from torch import nn
from transformers import AutoModel, AutoModelForSequenceClassification, AutoTokenizer, pipeline

import src.evaluate as ev
from src.data import get_data
from src.evaluate import ROOT, log_result, metrics
from src.preprocess import build_pipeline
from src.train import best_threshold, fit_with_threshold, set_seed

set_seed(42)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(DEVICE)

/usr/local/lib/python3.13/dist-packages/tashaphyne/stemming.py:106: SyntaxWarning: invalid escape sequence '\w'
  self.token_pat = re.compile(u"[^\w\u064b-\u0652']+", re.UNICODE)
/usr/local/lib/python3.13/dist-packages/pyarabic/araby.py:274: SyntaxWarning: invalid escape sequence '\w'
  TOKEN_PATTERN = re.compile(u"([^\w\u0670\u064b-\u0652']+)", re.UNICODE)
/usr/local/lib/python3.13/dist-packages/pyarabic/araby.py:276: SyntaxWarning: invalid escape sequence '\w'
  TOKEN_PATTERN_SPLIT = re.compile(u"([\w\u0670\u064b-\u0652']+)", re.UNICODE)
/usr/local/lib/python3.13/dist-packages/pyarabic/araby.py:281: SyntaxWarning: invalid escape sequence '\s'
  ARABIC_STRING = re.compile(u"([^\u0600-\u0652%s%s%s\s\d])" \
/usr/local/lib/python3.13/dist-packages/pyarabic/araby.py:1237: SyntaxWarning: invalid escape sequence '\s'
  u"(?<=\s(%s|%s))%s" % (WAW, YEH, FATHA), \
/usr/local/lib/python3.13/dist-packages/pyarabic/araby.py:1450: SyntaxWarning: invalid escape sequence '\s'
  text = re.sub(u"(?<=[

cuda


In [4]:
train, val, test, report = get_data()
y_train, y_val, y_test = train["sarcasm"].values, val["sarcasm"].values, test["sarcasm"].values
print(f"train: {len(train)} | val: {len(val)} | test: {len(test)} | test sarcasm rate: {y_test.mean():.3f} (val: {y_val.mean():.3f})")

VAL_RESULTS = ev.RESULTS                      # results/results.csv  (read only)
MODELS = ROOT / "models"
TEST_DIR = ROOT / "test"
(TEST_DIR / "predictions").mkdir(parents=True, exist_ok=True)
ev.RESULTS = TEST_DIR / "results_test.csv"    # log_result now upserts into test/results_test.csv
print("val results :", VAL_RESULTS)
print("test results:", ev.RESULTS)

train: 10781 | val: 1255 | test: 3000 | test sarcasm rate: 0.274 (val: 0.173)
val results : /content/Arabic-sarcasm/results/results.csv
test results: /content/Arabic-sarcasm/test/results_test.csv


In [5]:
BASE = ["html", "mentions", "hashtags"]
EMOJI = ["emoji"]
NORM = ["diacritics", "kashida", "letters", "elongation"]
LIGHT = ["light_stem"]
URLS = ["urls"]

COMBOS = {
    # same names / definitions as 02_baseline (so rows are comparable in results.csv)
    "base":                      BASE,
    "emoji":                     BASE + EMOJI,
    "norm":                      BASE + NORM,
    "light_stem":                BASE + LIGHT,
    "emoji + norm":              BASE + EMOJI + NORM,
    "emoji + light_stem":        BASE + EMOJI + LIGHT,
    "norm + light_stem":         BASE + NORM + LIGHT,
    "emoji + norm + light_stem": BASE + EMOJI + NORM + LIGHT,
    "urls kept":                 BASE,
    # added for PLMs: URLs -> <URL>, no stemming, AraBERT's own preprocessor
    "urls replaced":             BASE + URLS,
    "emoji + urls replaced":     BASE + URLS + EMOJI,
    "minimal":                   ["html", "urls", "mentions"],   # what 05_finetune uses
    "arabert":                   None,
}

In [6]:
def make_prep(steps):
    if steps is None:
        from arabert.preprocess import ArabertPreprocessor
        return ArabertPreprocessor(model_name="aubmindlab/bert-base-arabertv02-twitter").preprocess
    return build_pipeline(steps)


def prepare(frame, steps):
    prep = make_prep(steps)
    texts = [prep(tweet) for tweet in frame["tweet"]]
    return [t if t.strip() else raw for t, raw in zip(texts, frame["tweet"])]


try:
    make_prep(None)("هذا مثال 😂 @user #وسم_تجريبي http://t.co/x")   # fails here, not mid-run, if arabert is incompatible
except Exception as error:
    print("arabert preprocessing skipped:", error)
    COMBOS.pop("arabert")

print(list(COMBOS))
print("base       :", prepare(val.head(4), COMBOS["base"])[3][:100])
print("urls kept == base (BASE has no urls step), so those two rows are expected to match")

arabert preprocessing skipped: module 'emoji' has no attribute 'UNICODE_EMOJI'
['base', 'emoji', 'norm', 'light_stem', 'emoji + norm', 'emoji + light_stem', 'norm + light_stem', 'emoji + norm + light_stem', 'urls kept', 'urls replaced', 'emoji + urls replaced', 'minimal']
base       : "ريكي مارتن يخطب الرسام السوري جوان يوسفhttps://t.co/vm0NYjJIRF"
urls kept == base (BASE has no urls step), so those two rows are expected to match


In [7]:
def slug(name):
    return name.replace(" + ", "+").replace(" ", "_")


TOP_K = 1      # best preprocessings per model (D2 always uses the best preprocessing with all its seeds)

val_results = pd.read_csv(VAL_RESULTS)


def best(models, k=TOP_K):
    rows = val_results[val_results["model"].isin(models)]
    return rows.sort_values("val_f1", ascending=False).groupby("model").head(k)


def log_test(r, proba, threshold, source, val_scores=None):
    scores = metrics(y_test, proba, threshold)
    vals = val_scores or {c[4:]: r[c] for c in r.index if c.startswith("val_")}

    row = {"model": r["model"], "preprocessing": r["preprocessing"], "seed": int(r["seed"]),
           "threshold": float(threshold), "source": source}
    row.update({f"val_{k}": float(v) for k, v in vals.items()})
    row.update({f"test_{k}": float(v) for k, v in scores.items()})
    log_result(**row)

    pd.DataFrame({"id": test["id"], "y_true": y_test, "y_pred": (np.asarray(proba) >= threshold).astype(int),
                  "proba": proba}).to_csv(
        TEST_DIR / "predictions" / f"{r['model']}_{slug(r['preprocessing'])}_seed{int(r['seed'])}.csv", index=False)

    print(f"{r['model']:<28} {r['preprocessing']:<26} seed {int(r['seed'])}  "
          f"val_f1={row['val_f1']:.4f} -> test_f1={scores['f1']:.4f}  pr_auc={scores['pr_auc']:.4f}  "
          f"(thr {threshold:.3f}, {source})")

## B1 — zero-shot NLI

In [8]:
HYPOTHESES = {
    "ar": (["ساخرة", "جادة"],          "هذه التغريدة {}."),
    "fr": (["sarcastique", "sérieux"], "Ce tweet est {}."),
    "en": (["sarcastic", "serious"],   "This tweet is {}."),
}

zs = pipeline(
    "zero-shot-classification",
    model="MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7",
    device=0 if DEVICE == "cuda" else -1,
)

for _, r in best([f"zeroshot_nli_{lang}" for lang in HYPOTHESES]).iterrows():
    labels, template = HYPOTHESES[r["model"].split("_")[-1]]
    texts = prepare(test, COMBOS[r["preprocessing"]])
    out = zs(texts, candidate_labels=labels, hypothesis_template=template, batch_size=32)
    proba = np.array([dict(zip(o["labels"], o["scores"]))[labels[0]] for o in out])
    log_test(r, proba, r["threshold"], "pretrained")

config.json:   0%|          | 0.00/1.09k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  558MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/467 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 16.3MB            

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

zeroshot_nli_fr              emoji + urls replaced      seed 42  val_f1=0.4218 -> test_f1=0.4879  pr_auc=0.4226  (thr 0.724, pretrained)
zeroshot_nli_en              emoji + urls replaced      seed 42  val_f1=0.4198 -> test_f1=0.4753  pr_auc=0.4240  (thr 0.854, pretrained)
zeroshot_nli_ar              emoji + urls replaced      seed 42  val_f1=0.4174 -> test_f1=0.4674  pr_auc=0.4158  (thr 0.805, pretrained)


## B2 — sentiment model + rule "negative ⇒ sarcastic"

In [9]:
CAMEL = "CAMeL-Lab/bert-base-arabic-camelbert-da-sentiment"
tok_s = AutoTokenizer.from_pretrained(CAMEL)
clf = AutoModelForSequenceClassification.from_pretrained(CAMEL).to(DEVICE).eval()
NEG = [i for i, label in clf.config.id2label.items() if "neg" in label.lower()][0]


@torch.no_grad()
def sentiment_proba(texts, batch=64, max_len=128):
    order = np.argsort([len(t) for t in texts])
    proba = np.zeros((len(texts), clf.config.num_labels), dtype=np.float32)
    for i in range(0, len(texts), batch):
        idx = order[i:i + batch]
        enc = tok_s([texts[j] for j in idx], padding=True, truncation=True, max_length=max_len, return_tensors="pt").to(DEVICE)
        proba[idx] = torch.softmax(clf(**enc).logits.float(), dim=-1).cpu().numpy()
    return proba


for _, r in best(["camelbert_sent_hard", "camelbert_sent_soft"]).iterrows():
    proba = sentiment_proba(prepare(test, COMBOS[r["preprocessing"]]))
    score = (proba.argmax(1) == NEG).astype(float) if r["model"].endswith("hard") else proba[:, NEG]
    log_test(r, score, r["threshold"], "pretrained")

config.json:   0%|          | 0.00/841 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/86.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/305k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  436MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  436MB            

model.safetensors: downloading bytes:           |  0.00B            

camelbert_sent_soft          norm                       seed 42  val_f1=0.5000 -> test_f1=0.5240  pr_auc=0.5026  (thr 0.951, pretrained)
camelbert_sent_hard          emoji + norm               seed 42  val_f1=0.4607 -> test_f1=0.5321  pr_auc=0.3729  (thr 0.500, pretrained)


## Frozen AraBERT-twitter (shared by D1 and D2)

In [10]:
NAME = "aubmindlab/bert-base-arabertv02-twitter"
MAX_LEN, BATCH = 96, 64

tok = AutoTokenizer.from_pretrained(NAME)
encoder = AutoModel.from_pretrained(NAME).to(DEVICE).eval()
for p in encoder.parameters():
    p.requires_grad = False
HIDDEN = encoder.config.hidden_size

config.json:   0%|          | 0.00/667 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/476 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/751k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.25M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  541MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: aubmindlab/bert-base-arabertv02-twitter
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
pooler.dense.bias                          | MISSING    | 
pooler.dense.weight                        | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## D1 — `[CLS]` / mean pooling → logistic regression

Loads `models/arabert_frozen_{cls,mean}_lr_<prep>.joblib`; if absent, refits that single configuration (same recipe as the val notebook).

In [11]:
@torch.no_grad()
def embed(texts):
    order = np.argsort([len(t) for t in texts])
    cls = np.zeros((len(texts), HIDDEN), dtype=np.float32)
    mean = np.zeros((len(texts), HIDDEN), dtype=np.float32)
    for i in range(0, len(texts), BATCH):
        idx = order[i:i + BATCH]
        enc = tok([texts[j] for j in idx], padding=True, truncation=True, max_length=MAX_LEN, return_tensors="pt").to(DEVICE)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=DEVICE == "cuda"):
            hidden = encoder(**enc).last_hidden_state.float()
        mask = enc["attention_mask"].unsqueeze(-1).float()
        cls[idx] = hidden[:, 0].cpu().numpy()
        mean[idx] = ((hidden * mask).sum(1) / mask.sum(1).clamp(min=1)).cpu().numpy()
    return cls, mean


def fit_lr(X_train, X_val):
    best_fit = None
    for C in (0.01, 0.1, 1.0):
        model = make_pipeline(StandardScaler(), LogisticRegression(C=C, class_weight="balanced", max_iter=2000))
        threshold, seconds = fit_with_threshold(model, X_train, y_train, X_val, y_val)
        scores = metrics(y_val, model.predict_proba(X_val)[:, 1], threshold)
        if best_fit is None or scores["f1"] > best_fit["scores"]["f1"]:
            best_fit = dict(C=C, model=model, threshold=threshold, scores=scores)
    return best_fit


memo = {}
for _, r in best(["arabert_frozen_cls_lr", "arabert_frozen_mean_lr"]).iterrows():
    prep, k = r["preprocessing"], (0 if "_cls_" in r["model"] else 1)
    steps = COMBOS[prep]
    path = MODELS / f"{r['model']}_{slug(prep)}.joblib"

    val_scores = None
    if path.exists():
        ck, source = joblib.load(path), "loaded"
    else:
        if prep not in memo:
            memo[prep] = embed(prepare(train, steps)), embed(prepare(val, steps))
        tr, va = memo[prep]
        fit = fit_lr(tr[k], va[k])
        ck = {"model": fit["model"], "threshold": float(fit["threshold"]), "C": fit["C"], "pooling": ("cls", "mean")[k],
              "encoder": NAME, "max_len": MAX_LEN, "preprocessing": prep, "steps": steps}
        MODELS.mkdir(exist_ok=True)
        joblib.dump(ck, path)
        source, val_scores = "retrained", fit["scores"]

    X = embed(prepare(test, steps))[k]
    log_test(r, ck["model"].predict_proba(X)[:, 1], ck["threshold"], source, val_scores)

arabert_frozen_cls_lr        minimal                    seed 42  val_f1=0.6420 -> test_f1=0.6045  pr_auc=0.6077  (thr 0.654, retrained)
arabert_frozen_mean_lr       emoji + urls replaced      seed 42  val_f1=0.6384 -> test_f1=0.6004  pr_auc=0.6025  (thr 0.641, retrained)


## D2 — token states → BiLSTM + attention pooling

Best preprocessing on val, all its seeds. Loads `models/arabert_frozen_bilstm_att_<prep>_seed<n>.pt`; if absent, retrains that seed only.

In [12]:
class AttentionPooling(nn.Module):
    """alpha_i = softmax(v^T tanh(W h_i)),  c = sum_i alpha_i h_i  (padding masked)"""

    def __init__(self, dim, hidden=128):
        super().__init__()
        self.W = nn.Linear(dim, hidden)
        self.v = nn.Linear(hidden, 1, bias=False)

    def forward(self, h, mask):
        scores = self.v(torch.tanh(self.W(h))).squeeze(-1)
        alpha = torch.softmax(scores.masked_fill(~mask, -1e9), dim=-1)
        return (alpha.unsqueeze(-1) * h).sum(1), alpha


class BiLSTMAttention(nn.Module):
    def __init__(self, in_dim=HIDDEN, hidden=128, att=128, dropout=0.3):
        super().__init__()
        self.drop = nn.Dropout(dropout)
        self.lstm = nn.LSTM(in_dim, hidden, batch_first=True, bidirectional=True)
        self.att = AttentionPooling(2 * hidden, att)
        self.out = nn.Linear(2 * hidden, 2)

    def forward(self, x, mask):
        packed = nn.utils.rnn.pack_padded_sequence(
            self.drop(x), mask.sum(1).cpu(), batch_first=True, enforce_sorted=False)
        h, _ = self.lstm(packed)
        h, _ = nn.utils.rnn.pad_packed_sequence(h, batch_first=True, total_length=x.size(1))
        context, alpha = self.att(self.drop(h), mask)
        return self.out(context), alpha


@torch.no_grad()
def token_states(texts):
    L = max(len(ids) for ids in tok(texts, truncation=True, max_length=MAX_LEN)["input_ids"])
    states = torch.zeros(len(texts), L, HIDDEN, dtype=torch.float16)
    mask = torch.zeros(len(texts), L, dtype=torch.bool)
    order = np.argsort([len(t) for t in texts])
    for i in range(0, len(texts), BATCH):
        idx = order[i:i + BATCH]
        enc = tok([texts[j] for j in idx], padding=True, truncation=True, max_length=MAX_LEN, return_tensors="pt").to(DEVICE)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=DEVICE == "cuda"):
            hidden = encoder(**enc).last_hidden_state
        width, rows = hidden.size(1), torch.as_tensor(idx)
        states[rows, :width] = hidden.half().cpu()
        mask[rows, :width] = enc["attention_mask"].bool().cpu()
    return states, mask


def to_device(states, mask, idx):
    x, m = states[idx], mask[idx]
    width = int(m.sum(1).max())
    return x[:, :width].to(DEVICE).float(), m[:, :width].to(DEVICE)


@torch.no_grad()
def predict(model, states, mask, batch=256):
    model.eval()
    out = []
    for i in range(0, len(states), batch):
        x, m = to_device(states, mask, torch.arange(i, min(i + batch, len(states))))
        out.append(torch.softmax(model(x, m)[0], dim=-1)[:, 1].cpu())
    return torch.cat(out).numpy()


counts = train["sarcasm"].value_counts().sort_index()
WEIGHTS = torch.tensor(len(train) / (2 * counts.values), dtype=torch.float).to(DEVICE)


def fit_head(model, train_data, val_data, epochs=15, patience=3, lr=1e-3, batch=64):
    """Only used when a saved head is missing. Same recipe as the val notebook."""
    states, mask = train_data
    y = torch.as_tensor(y_train, dtype=torch.long)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    loss_fn = nn.CrossEntropyLoss(weight=WEIGHTS)
    best_f1, best_state, wait = -1.0, None, 0

    for epoch in range(epochs):
        model.train()
        perm = torch.randperm(len(states))
        for i in range(0, len(perm), batch):
            idx = perm[i:i + batch]
            x, m = to_device(states, mask, idx)
            loss = loss_fn(model(x, m)[0], y[idx].to(DEVICE))
            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

        proba = predict(model, *val_data)
        f1 = metrics(y_val, proba, best_threshold(y_val, proba))["f1"]
        if f1 > best_f1:
            best_f1, best_state, wait = f1, copy.deepcopy(model.state_dict()), 0
        else:
            wait += 1
            if wait >= patience:
                break
    model.load_state_dict(best_state)

In [13]:
d2 = val_results[val_results["model"] == "arabert_frozen_bilstm_att"]
best_prep = d2.loc[d2["val_f1"].idxmax(), "preprocessing"]            # chosen on val
steps = COMBOS[best_prep]
print("best preprocessing on val:", best_prep, "| seeds:", sorted(d2[d2["preprocessing"] == best_prep]["seed"].tolist()))

Xte = token_states(prepare(test, steps))
Xtr = Xva = None

for _, r in d2[d2["preprocessing"] == best_prep].iterrows():
    seed = int(r["seed"])
    path = MODELS / f"arabert_frozen_bilstm_att_{slug(best_prep)}_seed{seed}.pt"

    val_scores = None
    if path.exists():
        ck, source = torch.load(path, map_location=DEVICE), "loaded"
    else:
        if Xtr is None:
            Xtr, Xva = token_states(prepare(train, steps)), token_states(prepare(val, steps))
        set_seed(seed)
        head = BiLSTMAttention().to(DEVICE)
        fit_head(head, Xtr, Xva)
        proba_val = predict(head, *Xva)
        threshold = best_threshold(y_val, proba_val)
        ck = {"state_dict": {k: v.cpu() for k, v in head.state_dict().items()}, "threshold": float(threshold),
              "head": dict(in_dim=HIDDEN, hidden=128, att=128, dropout=0.3), "encoder": NAME, "max_len": MAX_LEN,
              "preprocessing": best_prep, "steps": steps, "seed": seed}
        MODELS.mkdir(exist_ok=True)
        torch.save(ck, path)
        source, val_scores = "retrained", metrics(y_val, proba_val, threshold)

    head = BiLSTMAttention(**ck["head"]).to(DEVICE)
    head.load_state_dict(ck["state_dict"])
    log_test(r, predict(head, *Xte), ck["threshold"], source, val_scores)

best preprocessing on val: minimal | seeds: [42, 43, 44]
arabert_frozen_bilstm_att    minimal                    seed 42  val_f1=0.6887 -> test_f1=0.5927  pr_auc=0.6179  (thr 0.754, retrained)
arabert_frozen_bilstm_att    minimal                    seed 43  val_f1=0.6667 -> test_f1=0.6240  pr_auc=0.6300  (thr 0.755, retrained)
arabert_frozen_bilstm_att    minimal                    seed 44  val_f1=0.6535 -> test_f1=0.6124  pr_auc=0.6345  (thr 0.677, retrained)


## Summary

In [14]:
out = pd.read_csv(ev.RESULTS)
view = out[["model", "preprocessing", "seed", "source", "threshold", "val_f1", "test_f1", "test_pr_auc", "test_macro_f1",
            "test_precision", "test_recall"]].copy()
view["f1_drop"] = view["val_f1"] - view["test_f1"]
view.sort_values("test_f1", ascending=False).round(4)

,model,preprocessing,seed,source,threshold,val_f1,test_f1,test_pr_auc,test_macro_f1,test_precision,test_recall,f1_drop
8,arabert_frozen_bilstm_att,minimal,43,retrained,0.7554,0.6667,0.6240,0.6300,0.7316,0.5742,0.6833,0.0426
9,arabert_frozen_bilstm_att,minimal,44,retrained,0.6774,0.6535,0.6124,0.6345,0.7260,0.5748,0.6553,0.0411
5,arabert_frozen_cls_lr,minimal,42,retrained,0.6545,0.6420,0.6045,0.6077,0.7159,0.5499,0.6711,0.0375
6,arabert_frozen_mean_lr,emoji + urls replaced,42,retrained,0.6414,0.6384,0.6004,0.6025,0.7103,0.5370,0.6809,0.0380
7,arabert_frozen_bilstm_att,minimal,42,retrained,0.7541,0.6887,0.5927,0.6179,0.7156,0.5719,0.6151,0.0960
4,camelbert_sent_hard,emoji + norm,42,pretrained,0.5000,0.4607,0.5321,0.3729,0.5994,0.3964,0.8088,-0.0713
3,camelbert_sent_soft,norm,42,pretrained,0.9506,0.5000,0.5240,0.5026,0.6707,0.5172,0.5311,-0.0240
0,zeroshot_nli_fr,emoji + urls replaced,42,pretrained,0.7244,0.4218,0.4879,0.4226,0.5839,0.3786,0.6857,-0.0661
1,zeroshot_nli_en,emoji + urls replaced,42,pretrained,0.8536,0.4198,0.4753,0.4240,0.6198,0.4254,0.5384,-0.0555
2,zeroshot_nli_ar,emoji + urls replaced,42,pretrained,0.8055,0.4174,0.4674,0.4158,0.5805,0.3737,0.6236,-0.0499


In [15]:
d2_test = out[out["model"] == "arabert_frozen_bilstm_att"]
print("D2 over seeds (mean ± std):")
print(d2_test[["val_f1", "test_f1", "test_pr_auc", "test_macro_f1"]].agg(["mean", "std"]).round(4))

D2 over seeds (mean ± std):
      val_f1  test_f1  test_pr_auc  test_macro_f1
mean  0.6696   0.6097       0.6275         0.7244
std   0.0178   0.0158       0.0086         0.0081


### Save to disk
Colab wipes `/content` when the session ends: download `test/` (metrics + predictions) and `models/` if anything was retrained.

In [16]:
if "google.colab" in sys.modules:
    from google.colab import files
    !zip -qr /content/test_partF.zip test models
    files.download("/content/test_partF.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>